# Assignment 5: Building a ReAct Agent from Scratch
<b style="color:red;bold;">Due date: April 1st, 2026</b>

## 0. Preliminary (Please Read this Carefully)

### Libraries

The assignment mainly relies on a library developped by Huggingface:

- `transformers`: A comprehensive library that provides state-of-the-art pre-trained models for Natural Language Processing (NLP) and other tasks, offering easy-to-use interfaces for implementing, and fine-tuning a wide range of transformer-based architectures.


In case you are unfamiliar with these libraries, you can consult these resources:

- [Stanford's CS224N transformers tutorial](https://colab.research.google.com/drive/1pxc-ehTtnVM72-NViET_D2ZqOlpOi2LH?usp=sharing)


### Model

In this assignment you will use the Qwen3-4B model. It roughly has 4B parameters. Compared to other models of similar size, it performs better, with improved reasoning and instruction-following capabilities. You can find the model on Huggingface's [hub](https://huggingface.co/Qwen/Qwen3-4B).


### Execution Environment

It is strongly recommended to use `Google Colab` with the hardware accelerator set to `T4 GPU`, as it enables faster computation and allows the model to be loaded without encountering CUDA out-of-memory errors.

### Implementation Guide
- You are tasked with filling in the sections marked as **TODO** and indicated by **ellipses (...)**.
- No external libraries are needed.
- Please ensure that the output of the final cell is preserved, as it is required to observe the model's behavior and for marking purposes.

---

## 1. The Challenges of Standalone LLMs
Traditional "zero-shot" LLMs are powerful. However, they face four major limitations:

1.  **Hallucination & Outdated Knowledge:** LLMs predict the next statistically likely word, not the truth. They do not "know" facts, often making up answers (hallucinations), and cannot access real-time data beyond their training cutoff.

2. **Stateless by default:** They don't remember past steps automatically. Each call must include the **needed context/history** (or an external memory) so the model knows what happened before.

3. **Not autonomous on their own:** An LLM wont independently plan, run tools, monitor progress, and retry failures.

3.  **Poor Logic & Math:** LLMs are text predictors, not calculators. They struggle with precise arithmetic and multi-step logic, often failing at tasks that a simple calculator could solve instantly.


## 2. What is Agentic AI?
 Agentic AI helps LLMs pursue goals autonomously. Traditional LLMs are passive: you give input, they give output. Agentic AI is active. In fact, instead of just *answering*, an Agent is designed to *act*. It can use tools, query databases, execute code, and perform multi-step reasoning to achieve a goal.

### Why we need agents
It enables LLMs to show better and more accurate responses in the following scenarios:
- **multiple steps** (plan → compute → verify → format),
- **reliable calculations** (where a tool is safer than “mental math”),
- **interaction with external systems** (files, data, web, APIs),
- **error recovery** (retrying when something fails).

## 3. The ReAct Framework (A method to build an Agent)
**ReAct** (Reasoning + Acting) is a prompting framework to achive **Agentic AI** that allows LLMs to solve complex problems by interleaving **Thought**, **Action**, and **Observation**.

Without a framework like ReAct, an LLM doesn't know how to be an agent. It might try to do everything in one breath or hallucinate an answer. ReAct forces the LLM into a specific loop that simulates a thought process:

* **Thought:** The agent reasons about the current state of the problem. ("I need to calculate the square root first.")
* **Action:** The agent performs a specific action, such as calling a tool. ("Run Python code: `sqrt(144)`")
* **Observation:** The agent waits for the external tool to return a result. ("Result: 12")
* **Repeat:** The agent uses this new observation to continue its reasoning.


## 4. Assignment Goal
In this notebook, you will build a functional **ReAct Agent** from scratch, capable of solving multi-step reasoning problems that a standard LLM would fail at.




### Step 1: Loading the LLM & Inference Function
Here we load the **Qwen3-4B** model. We also define a helper function, `call_llm`, which performs a simple inference on the prompt. It takes the entire list of message history, formats it into a prompt the model understands, and returns the model's text response.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import sys
from io import StringIO
import contextlib
import re

model_name = "Qwen/Qwen3-4B"

print(f"Loading {model_name}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map=device,
    torch_dtype=torch.float16 if device != "cpu" else torch.float32,
)

def call_llm(messages):
    """Sends a list of messages to the LLM and returns the text response."""

    # Apply the model's specific chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the text and return a pytorch tensor in the device mode
    model_inputs = tokenizer(text, return_tensors="pt").to(device)


    # generate the LLM's response:
    # 1. Max generated token length should not be more than 512 tokens
    # 2. Enable deterministic behavior of LLM by setting proper arguments
    generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=512,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)
    # Setting the temperature to zero makes an LLM deterministic, producing the most probable output with minimal randomness.

    # Decode only the new tokens and return the decoded string
    new_tokens = generated_ids[0][model_inputs.input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


Loading Qwen/Qwen3-4B...
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

### Step 2: Defining the Tool (Don't modify this cell)
Here, we define a `PythonTool`. This class allows the Agent to execute real Python code. It captures the standard output (`stdout`) so the Agent can "see" the result of its calculations.

In [3]:
class PythonTool:
    def __init__(self):
        self.locals = {}

    def run(self, code_str):
        print("-" * 50)
        print(f"\n[Running Code]:\n{code_str}\n")
        io_buffer = StringIO()
        try:
            with contextlib.redirect_stdout(io_buffer):
                exec(code_str, {}, self.locals)
            return io_buffer.getvalue().strip()
        except Exception as e:
            return f"Error: {e}"


### The ReAct Agent Loop:
**Purpose:** implement the **agent controller** that repeatedly:
1) asks the LLM for the next step,  
2) detects whether it chose a tool action,  
3) runs the tool,  
4) feeds the observation back into the conversation,  
5) stops when the model is done.

In [4]:
import re


def run_agent(question, max_steps=10):
    tool = PythonTool()

    # TODO:Here, we define the system instructions for the agent. Read the instructions to see what the agent is expected to do.
    # However, DO NOT MODIFY THE INSTRUCTIONS.
    system_instructions = """You are a ReAct-based math assistant. Use this loop until the problem is solved:

**Thought**: Briefly state what you will do next (do NOT reveal full step-by-step hidden reasoning).
**Action**: Choose exactly one:
  1) Tool: run Python code in <python></python> tag to compute anything arithmetic/algorithmic and then `print` to see the result:
     <python>
     # write code
     print(result)
     </python>
  2) Final: provide the final answer in the format:
     Final Answer: ...

**Observation**: (system/tool output)
  - If you used Tool, stop your message after the tool block.
  - Wait for the Observation, then continue the loop.

Rules:
- DO NOT DO MATH ON YOUR OWN
- Do not output both a Tool action and a Final answer in the same turn.
"""


    messages = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": question}
    ]

    print(f"--- Question: {question} ---")

    for step in range(max_steps):
        # Get LLM Response
        response = call_llm(messages)

        print(f"\n[Step {step+1}] Agent:\n{response}")
        # append the assistant's response to the messages list
        messages.append({"role": "assistant", "content": response})

        # Check for Final Answer
        if "Final Answer:" in response:
            # Extract the final answer
            final_answer = response.split("Final Answer:")[-1].strip()
            print(f"\n[Final Answer]: {final_answer}")
            break

        # Check for Python tool call
        elif "<python>" in response and "</python>" in response:
            # Extract code between python tags
            import re
            python_match = re.search(r'<python>(.*?)</python>', response, re.DOTALL)
            if python_match:
                code = python_match.group(1).strip()

                # Run the tool
                result = tool.run(code)

                # Add observation to messages
                observation_msg = f"**Observation**: {result}"
                messages.append({"role": "user", "content": observation_msg})
                print(f"\n[Observation]: {result}")
            else:
                # Malformed python block
                messages.append({"role": "user", "content": "Please provide valid Python code in <python></python> tags."})
                print("\n[System]: Malformed Python block detected.")

        else:
            # Neither final answer nor tool call found
            messages.append({"role": "user", "content": "Please continue with either a Tool action using <python></python> tags or provide a Final Answer."})
            print("\n[System]: Prompting agent to continue...")

    print("\n[System]: Max steps reached without final answer.")

## Test the Framework

- In the following cell, you should observe that the model first reasons about the problem and then writes a Python function and `call the tool` to compute the fifth root of the number, which equals to 2017. Model adds this to its `observation`. The U.S. president at that time was Donald Trump (Model knows this based on its prior knowledge and report it in the `thought mode`). Taking his last name, **“Trump”**, which contains 5 letters, the model agains `call the tool` to compute:
$(\log_{10}(5))^{\frac{1}{4}} = 0.9143$.

- Your code (model) should `call the tool` at least once. However, ideally, as explained above, it should `call the tool` twice.

In [5]:
problem = """
Calculate the fifth root of 33,383,317,356,629,857. Interpret the resulting integer as a year AD.
Identify the U.S. president at that time. Then, take the number of letters in their last name, compute the base-10 logarithm of that number, and calculate the fourth root of the result. Finally, report the exact floating-point value.
"""
run_agent(problem)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- Question: 
Calculate the fifth root of 33,383,317,356,629,857. Interpret the resulting integer as a year AD.
Identify the U.S. president at that time. Then, take the number of letters in their last name, compute the base-10 logarithm of that number, and calculate the fourth root of the result. Finally, report the exact floating-point value.
 ---

[Step 1] Agent:
<think>
Okay, let's tackle this problem step by step. First, I need to find the fifth root of 33,383,317,356,629,857. Hmm, how do I do that? Well, maybe I can use Python's math library for that. Let me check. The fifth root of a number x is the same as x raised to the power of 1/5. So I'll compute that.

Wait, but I should make sure that the number is indeed a perfect fifth power. Let me try calculating it. Let me write a Python code snippet to compute the fifth root. Oh, but wait, maybe I can use the ** operator. Let me test that. Let me input the number and take the 1/5 power. Let me see.

Once I get the fifth root, I nee

## Rubric:

- call_llm: /3
- run_agent: /7
- output showing that at least one time `python tool` is called: /2

## Total: /12
